# Teacher–student divergence (Overlap Top-K) + bootstrap confidence intervals
### Experiments #1 and #2 · SDAR-4B-Chat-b32 · **inference only, no training, no LoRA**

Two additions to *The Compression Floor*, both cheap and both aimed at the paper's
weakest-evidenced claims.

**Part A — Overlap Top-K (§7–§10).** The paper currently asserts that the privileged teacher
supplies almost no gradient signal because "conditioning on a reference solution barely moved
the teacher's distribution." That is an inspection, not a measurement. This notebook measures
it, using the metric Luo et al. (2026, arXiv:2606.18195) define for exactly this purpose:

$$\mathcal{M}_{\text{overlap}} = \frac{1}{|\mathcal{K}_t|}\sum_{i\in\mathcal{K}_t}
\frac{|\mathcal{P}^{i,\text{Top-}K}_{\text{student}} \cap \mathcal{P}^{i,\text{Top-}K}_{\text{teacher}}|}{K}$$

At every denoising sub-step, over the positions actually being revealed, we compare the
student's and teacher's top-$K$ vocabulary sets. Overlap near 1 means the teacher is telling
the student what it already believes, so the KL target is ~0 and no learning can occur.

We measure **two teacher constructions on the same student states**:

| | Construction | Expectation |
|---|---|---|
| **A: privileged prefix** | reference solution appended to the teacher's prompt (our OPSD-1) | overlap near 1, i.e. no signal |
| **B: self-future lookahead** | teacher sees `future_reveal_k` of the student's own later tokens (our OPSD-2 / d-OPSD's construction) | overlap meaningfully below A |

If B is clearly below A, the paper gains a **positive** result to pair with its negative one:
the failure is specific to the teacher construction, not to on-policy self-distillation.

**Part B — Bootstrap CIs (§11).** Pure CPU, reads `results/floor_grid_per_problem.csv` from
the repo. Puts proper 95% intervals on every grid cell so the n=40 objection is answered with
statistics instead of an apology.

> **No LoRA is loaded anywhere in this notebook.** Teacher and student are the same frozen
> base model under different conditioning, which is the whole point of self-distillation.
> Dropping PEFT removes the adapter-state confound entirely.

## 1 · GPU check & Drive

In [ ]:
!nvidia-smi
import torch, platform
print("\nTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type ▸ A100."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 · Dependencies — pin transformers, remove torchao FIRST
`modeling_sdar.py` needs transformers 4.5x (Colab ships 5.x). We clone the repo, read its pin,
install it. **Critical ordering:** torchao is uninstalled *before* transformers is imported —
transformers caches `is_torchao_available()` at import time and then does a deferred torchao
import when loading the model, so removing it afterward is too late. **If the transformers
version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

# 0) remove torchao BEFORE any transformers import (stale Colab build breaks peft + transformers)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"
_spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    txt = open(_req).read()
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", txt, re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec, "(from requirements.txt)" if _spec != _FALLBACK else "(fallback)")

!pip -q install "{_spec}" "accelerate>=0.33" "peft>=0.12" "datasets>=2.20" \
                "bitsandbytes>=0.43" sentencepiece ninja packaging

# guard: pip may reinstall torchao transitively — remove again
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If the version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 4 · `flash_attn` → pure-PyTorch replacements (no install/compile)
Same mechanism that got TraDo loading: a meta-path finder serves numerically-equivalent
pure-PyTorch versions of the flash-attn kernels `modeling_sdar.py` imports (fused RMSNorm +
attention via SDPA), plus the `get_imports` patch and `LossKwargs` rename bridge.

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch
import torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old,_c in {"LossKwargs":["TransformersKwargs"]}.items():
    if not hasattr(_tu,_old):
        v=None
        for cand in _c:
            for mp in ("transformers.utils","transformers.processing_utils","transformers.modeling_utils","transformers"):
                try:
                    m=importlib.import_module(mp)
                    if hasattr(m,cand): v=getattr(m,cand); break
                except Exception: pass
            if v is not None: break
        setattr(_tu,_old, v if v is not None else type(_old,(dict,),{}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None, out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    if residual is not None:
        base = (x.float()+residual.float()) if residual_in_fp32 else (x+residual)
    else:
        base = x.float() if residual_in_fp32 else x
    nr = base
    xf = base.float()
    xn = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
    w = (1.0+weight) if zero_centered_weight else weight
    y = xn.to(xdt) * w
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    if residual is not None:
        base = (x.float()+residual.float()) if residual_in_fp32 else (x+residual)
    else:
        base = x.float() if residual_in_fp32 else x
    nr = base; xf = base.float(); mean = xf.mean(-1, keepdim=True)
    xn = (xf-mean)*torch.rsqrt((xf-mean).pow(2).mean(-1, keepdim=True)+eps)
    w = (1.0+weight) if zero_centered_weight else weight
    y = xn.to(xdt)*w
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k,v,nq):
    nk=k.shape[-2]
    if nk!=nq:
        r=nq//nk; k=k.repeat_interleave(r,dim=-2); v=v.repeat_interleave(r,dim=-2)
    return k,v

def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v=_expand_kv(k,v,q.shape[-2])
    o=F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                     is_causal=causal,scale=softmax_scale,dropout_p=0.0)
    return o.transpose(1,2)

def _flash_attn_qkvpacked_func(qkv,**kw):
    q,k,v=qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)

def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck=cu_seqlens_q.tolist(),cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi=q[cq[i]:cq[i+1]],k[ck[i]:ck[i+1]],v[ck[i]:ck[i+1]]
        ki,vi=_expand_kv(ki,vi,qi.shape[-2])
        oi=F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),ki.transpose(0,1).unsqueeze(0),
                                          vi.transpose(0,1).unsqueeze(0),is_causal=causal,
                                          scale=softmax_scale,dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)

def _pad_input(hs,idx,b,s):
    out=hs.new_zeros(b*s,hs.shape[-1]); out[idx]=hs; return out.view(b,s,-1)
def _unpad_input(hs,am,*a,**k):
    sl=am.sum(-1).to(torch.int32); idx=torch.nonzero(am.flatten(),as_tuple=False).flatten()
    h=hs.reshape(-1,hs.shape[-1])[idx]; cu=torch.zeros(sl.numel()+1,dtype=torch.int32,device=hs.device)
    cu[1:]=torch.cumsum(sl,0); return h,idx,cu,int(sl.max().item())
def _index_first_axis(x,idx): return x.reshape(-1,*x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self,hidden_size,eps=1e-6,**kw):
        super().__init__(); self.weight=torch.nn.Parameter(torch.ones(hidden_size)); self.eps=eps
    def forward(self,x,residual=None,prenorm=False,**kw):
        return _rms_norm_fn(x,self.weight,None,residual=residual,eps=self.eps,prenorm=prenorm)

_REG={"rms_norm_fn":_rms_norm_fn,"layer_norm_fn":_layer_norm_fn,"RMSNorm":_RMSNormModule,
      "LayerNorm":torch.nn.LayerNorm,"flash_attn_func":_flash_attn_func,
      "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,"flash_attn_varlen_func":_flash_attn_varlen_func,
      "pad_input":_pad_input,"unpad_input":_unpad_input,"index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a,**k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self,n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self,fn,path=None,target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn,self,is_package=True)
    def create_module(self,spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self,m): pass

try:
    import flash_attn; USE_FLASH_ATTN=True; print("Real flash_attn present — using it.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")
_t=torch.randn(2,4,8); _w=torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm matches reference. USE_FLASH_ATTN =", USE_FLASH_ATTN)


## 5 · torchao — already removed in §2 (kept as a no-op check)

In [ ]:
import importlib.util
print("torchao installed?", importlib.util.find_spec("torchao") is not None,
      "→ must be False (removed in §2 before transformers import).")


## 3 · Config

Defaults reproduce the paper's OPSD-1 schedule (student 2 tokens/step at block 32). The
projection printed below tells you the runtime before you commit. If it is too slow, raise
`state_stride` first: it subsamples denoising states, which costs precision on the mean but
not the qualitative A-vs-B gap.

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple
import os

@dataclass
class Config:
    drive_root: str = "/content/drive/MyDrive/teacher_divergence"

    # ---- model (frozen; no LoRA in this notebook) ----
    model_id: str = "JetLM/SDAR-4B-Chat-b32"
    load_in_4bit: bool = False

    # ---- data (identical fields to the OPSD-1 notebook so §7 is reusable verbatim) ----
    dataset_id: str = "zwhe99/DeepMath-103K"
    n_samples: int = 2000
    difficulty_max: Optional[int] = 5
    seed: int = 0
    col_question: str = "question"
    col_answer: str = "final_answer"
    col_difficulty: str = "difficulty"
    solution_cols: Tuple[str, ...] = ("r1_solution_1","r1_solution_2","r1_solution_3")
    n_eval_holdout: int = 8
    opsd_n: int = 800

    # ---- diffusion schedule (matches the paper's OPSD-1 student) ----
    block_size: int = 32
    student_steps_per_block: int = 16    # 2 tokens/step
    max_gen_tokens: int = 768            # 24 blocks; enough states, keeps the run ~2h
    temperature: float = 0.3             # the yield-safe setting from the paper
    top_p: float = 0.95
    noise_temp: float = 1.0

    # ---- the measurement ----
    n_problems: int = 20                 # problems to measure over
    topK: int = 20                       # K in Overlap Top-K (Luo et al. use K=20)
    future_reveal_k: int = 4             # teacher B: how many future positions to reveal
    state_stride: int = 1                # 1 = every denoising state; 2 = every other, etc.

cfg = Config()
os.makedirs(cfg.drive_root, exist_ok=True)
RESUME = True
LOG_PATH = os.path.join(cfg.drive_root, "overlap_topk.jsonl")

_sub = cfg.block_size // (cfg.block_size // cfg.student_steps_per_block)
_blocks = cfg.max_gen_tokens // cfg.block_size
_states = (_blocks * cfg.student_steps_per_block) // max(1, cfg.state_stride)
print(cfg, "\n")
print(f"per problem: {_blocks} blocks x {cfg.student_steps_per_block} sub-steps "
      f"= {_blocks*cfg.student_steps_per_block} states (stride {cfg.state_stride} -> {_states} measured)")
print(f"forwards/problem ~= {_blocks*cfg.student_steps_per_block} (student rollout) "
      f"+ 2 x {_states} (teachers A and B) = {_blocks*cfg.student_steps_per_block + 2*_states}")
print(f"at ~0.3 s/forward -> ~{(_blocks*cfg.student_steps_per_block + 2*_states)*0.3/60:.1f} min/problem"
      f"  |  {cfg.n_problems} problems -> ~{(_blocks*cfg.student_steps_per_block + 2*_states)*0.3*cfg.n_problems/3600:.1f} h")
print("\nRaise cfg.state_stride to cut this; the A-vs-B gap survives subsampling.")

## 4 · Load the frozen model, block-causal mask, logits

Byte-identical to the OPSD-1 notebook except that **PEFT is not used**: `model` is the base
model, and `model_logits(..., teacher=...)` no longer needs an adapter-disable context because
there is no adapter. Teacher and student differ only in what they are conditioned on.

In [ ]:
import torch, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("transformers:", transformers.__version__)
if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("5.0.0"):
    print("⚠️ transformers 5.x — modeling_sdar will fail. Re-run §2, restart, run from top.")

tok = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
quant = None
if cfg.load_in_4bit:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                               bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
_dt = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") else "torch_dtype"
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id, trust_remote_code=True, device_map="auto", quantization_config=quant,
    attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa", **{_dt: torch.bfloat16})
if base_model.config.pad_token_id is None:
    base_model.config.pad_token_id = tok.pad_token_id or tok.eos_token_id
print("Loaded:", type(base_model).__name__, "| attn:", getattr(base_model.config,"_attn_implementation","?"))

MASK_ID = getattr(tok, "mask_token_id", None) or 151669   # SDAR mask id
VOCAB = base_model.config.vocab_size; REAL_VOCAB = len(tok)
print("MASK_ID =", MASK_ID, "| VOCAB =", VOCAB, "| real vocab =", REAL_VOCAB, "| eos =", tok.eos_token_id)


### 6b · Discovery — how is block size / attention actually set? (verify before trusting)
Prints the model config and greps `configuration_sdar.py` / `modeling_sdar.py` for block-size and
attention handling. **What you want to see:** a `block_size`/`block_length` field in the config
equal to 32 (meaning the checkpoint's config drives block attention automatically), and an
attention path that builds a block mask. If block size is hardcoded to 4 or absent, the plain
forward may not match the trained block-32 attention — tell me what this prints.

In [ ]:
import glob, re
print("=== config fields mentioning block / attention ===")
for k,val in vars(base_model.config).items():
    if any(t in k.lower() for t in ("block","attn","window","diffus","mask")):
        print(f"  {k}: {val}")

for fname in ("configuration_sdar.py","modeling_sdar.py"):
    cand = glob.glob(os.path.expanduser(
        f"~/.cache/huggingface/modules/transformers_modules/**/{fname}"), recursive=True)
    if not cand: print(f"\n({fname} not found in cache)"); continue
    src = open(sorted(cand, key=os.path.getmtime)[-1]).read()
    print(f"\n===== {fname} — block/attention lines =====")
    for i,l in enumerate(src.splitlines()):
        if re.search(r"block_size|block_length|is_causal|attention_mask|block_diag|block_mask|def forward", l):
            print(f"  {l.strip()[:96]}")


In [ ]:
import torch, contextlib

# ---------------------------------------------------------------------------
# CRITICAL (unchanged from the OPSD notebooks): modeling_sdar.py has
# `_update_causal_mask` COMMENTED OUT. Whatever attention_mask we pass flows
# unmodified into SDARAttention. We build the block-causal / bidirectional-within-block
# pattern ourselves.
# ---------------------------------------------------------------------------
def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx - prompt_len) // block_size, idx)
    q_resp, k_resp = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    q_idx, k_idx = idx.unsqueeze(1), idx.unsqueeze(0)
    q_blk, k_blk = blk.unsqueeze(1), blk.unsqueeze(0)
    prompt_causal   = (~q_resp) & (k_idx <= q_idx)
    resp_see_prompt = q_resp & (~k_resp)
    resp_cross_blk  = q_resp & k_resp & (k_blk <= q_blk)
    return prompt_causal | resp_see_prompt | resp_cross_blk

model = base_model            # no LoRA: teacher and student are the same frozen weights
model.eval()

def model_logits(input_ids, prompt_len):
    seq_len = input_ids.shape[1]
    mask = build_block_causal_mask(seq_len, prompt_len, cfg.block_size, input_ids.device)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model(input_ids=input_ids, attention_mask=mask).logits

# vocab guard: mask padded dead slots + the MASK token itself
_inv = torch.zeros(VOCAB, dtype=torch.bool)
if REAL_VOCAB < VOCAB: _inv[REAL_VOCAB:] = True
_inv[MASK_ID] = True
INVALID_MASK = _inv.to(model.device)

def _append_masks(ids, b):
    return torch.cat([ids, torch.full((1,b), MASK_ID, dtype=ids.dtype, device=ids.device)], dim=1)

print("frozen model ready | no LoRA attached | invalid ids masked:", int(INVALID_MASK.sum()))

## 7 · Data: DeepMath-103K → verified pool
Keep the first R1 solution whose own boxed answer matches `final_answer` (the teacher must be
conditioned on a *correct* solution). Prompts use the model's **own chat template**
(`apply_chat_template`) rather than a hand-written one, since SDAR-Chat ships its own.

In [ ]:
def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i==-1: return None
    j = text.find("{", i)
    if j==-1: return None
    d=0
    for k in range(j,len(text)):
        if text[k]=="{": d+=1
        elif text[k]=="}":
            d-=1
            if d==0: return text[j+1:k]
    return None

def _norm(s):
    if s is None: return None
    s=str(s).strip().replace(" ","")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")): s=s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s=s[6:-1]
    return s

def answers_match(pred,gold):
    a,b=_norm(pred),_norm(gold)
    if a is None or b is None: return False
    if a==b: return True
    try: return abs(float(a)-float(b))<1e-6
    except Exception: return False

def repeat4(text):
    t=text.split()
    if len(t)<4: return 0.0
    g=[tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1-len(set(g))/len(g)

assert extract_boxed(r"x \boxed{12}, y \boxed{34}")=="34"
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
print("✅ answer helpers pass")


In [ ]:
from datasets import load_dataset
raw = load_dataset(cfg.dataset_id, split="train")
pool = raw.shuffle(seed=cfg.seed)
if cfg.difficulty_max is not None and cfg.col_difficulty in pool.column_names:
    pool = pool.filter(lambda e: e[cfg.col_difficulty] is not None and float(e[cfg.col_difficulty])<=cfg.difficulty_max)
    print(f"difficulty<= {cfg.difficulty_max}: {len(pool)} rows")
pool = pool.select(range(min(cfg.n_samples, len(pool))))

verified, dropped = [], 0
for e in pool:
    gold=e[cfg.col_answer]; sol=None
    for c in cfg.solution_cols:
        cand=e.get(c)
        if cand and answers_match(extract_boxed(str(cand)), gold): sol=str(cand); break
    if sol is None: dropped+=1; continue
    verified.append({"question":e[cfg.col_question], "solution":sol, "gold":str(gold)})
eval_holdout = verified[-cfg.n_eval_holdout:]; verified = verified[:-cfg.n_eval_holdout]
print(f"VERIFIED {len(verified)} | dropped {dropped} | held out {len(eval_holdout)}")

# This notebook MEASURES rather than trains, so it draws a fixed slice of the verified
# pool rather than iterating passes over it (unlike the OPSD-1 notebook this data cell
# was copied from, which had a cfg.target_passes field — this Config intentionally has
# no such field, since there are no training passes here).
opsd_pool = verified[:cfg.opsd_n]
print(f"verified pool: {len(opsd_pool)} problems available (target {cfg.opsd_n})")
print(f"this run measures overlap on the first cfg.n_problems={cfg.n_problems} of them")
if len(opsd_pool) < cfg.n_problems:
    print(f"WARNING only {len(opsd_pool)} verified (< cfg.n_problems={cfg.n_problems}); "
          f"raise cfg.n_samples in section 3 and re-run.")


In [ ]:
def build_student_prompt(q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer within \\boxed{{}}."}],
        tokenize=False, add_generation_prompt=True)

def build_teacher_prompt(q, sol, failure_note=""):
    content = (f"{q}\n\nHere is a reference solution:\n{sol}\n\n"
               f"After understanding the reference solution, solve the problem yourself.{failure_note}\n"
               f"Please reason step by step, and put your final answer within \\boxed{{}}.")
    return tok.apply_chat_template([{"role":"user","content":content}],
                                   tokenize=False, add_generation_prompt=True)

def correction_note(wrong, gold):
    return (f"\nNote: a previous attempt concluded \\boxed{{{wrong}}}, which is INCORRECT. "
            f"Identify the error and avoid it; the correct final answer is {gold}.")
print(build_student_prompt(verified[0]["question"])[:200])


## 8 · Student rollout that records its own denoising states

The student rolls out on-policy exactly as in OPSD-1: confidence-ordered reveal, `block_size //
student_steps_per_block` tokens per sub-step. The difference is bookkeeping. At each sub-step we
record

* `revealed_before` — the block-local positions already committed entering this state,
* `reveal_now` — the positions committed at this state (this is $\mathcal{K}_t$),
* `student_topk` — the student's top-$K$ vocabulary ids at those positions,

so the teachers can be evaluated later on **exactly** the states the student actually visited.
Nothing is recomputed from a hypothetical trajectory.

In [ ]:
import torch, torch.nn.functional as F

@torch.no_grad()
def student_rollout_with_states(question):
    """Returns (states, committed_blocks, prompt_len, text).

    states: list of dicts, one per denoising sub-step:
        {block, revealed_before: [pos], reveal_now: [pos], topk: LongTensor[len(reveal_now), K]}
    committed_blocks: list of LongTensor[block_size] the student actually committed
    """
    B = cfg.block_size
    per = B // cfg.student_steps_per_block
    s_ids = tok(build_student_prompt(question), return_tensors="pt").input_ids.to(model.device)
    prompt_len = s_ids.shape[1]

    states, committed_blocks = [], []
    produced = 0
    while produced < cfg.max_gen_tokens:
        work = _append_masks(s_ids, B)
        base = s_ids.shape[1]
        pos_all = list(range(base, base + B))
        remaining = set(range(B))
        committed = [None] * B
        revealed_before = []
        while remaining:
            lg = model_logits(work, prompt_len)[0, pos_all, :].float()
            lg = lg.masked_fill(INVALID_MASK, float("-inf"))
            logp = torch.log_softmax(lg, dim=-1)
            conf = logp.max(dim=-1).values
            if cfg.noise_temp > 0:                      # Gumbel noise on reveal ORDER only
                g = -torch.log(-torch.log(torch.rand_like(conf).clamp_min(1e-9)).clamp_min(1e-9))
                order_score = conf + cfg.noise_temp * g
            else:
                order_score = conf
            cand = sorted(remaining, key=lambda j: order_score[j].item(), reverse=True)[:per]

            topk_ids = torch.topk(logp[cand, :], cfg.topK, dim=-1).indices.cpu()
            states.append({"block": len(committed_blocks),
                           "revealed_before": list(revealed_before),
                           "reveal_now": list(cand),
                           "topk": topk_ids})

            for j in cand:
                probs = torch.softmax(lg[j] / max(cfg.temperature, 1e-6), dim=-1)
                tokid = int(torch.multinomial(probs, 1))
                committed[j] = tokid
                work[0, base + j] = tokid
                remaining.discard(j)
                revealed_before.append(j)

        blk = torch.tensor(committed, device=model.device)
        committed_blocks.append(blk)
        s_ids = work
        produced += B
        if any(int(t) == tok.eos_token_id for t in blk):
            break

    text = tok.decode(s_ids[0, prompt_len:], skip_special_tokens=True)
    return states, committed_blocks, prompt_len, text

print("student rollout ready | student reveals",
      cfg.block_size // cfg.student_steps_per_block, "tokens/step")

## 9 · Overlap Top-K for both teacher constructions

Both teachers are scored on the **same** student states. The only difference is conditioning.

**Teacher A (privileged prefix).** Prompt is `build_teacher_prompt(question, reference_solution)`,
so `prompt_len` differs from the student's and the response block sits at different absolute
indices. Response content is identical to the student's state.

**Teacher B (self-future lookahead).** Same prompt as the student. On top of the student's state
we additionally fill `future_reveal_k` positions that the student reveals *later* in its own
order, taken from the tokens it actually committed. These never overlap `reveal_now`, so the
teacher is never shown an answer to the position it is being scored on.

In [ ]:
@torch.no_grad()
def _topk_at(input_ids, prompt_len, abs_positions):
    lg = model_logits(input_ids, prompt_len)[0, abs_positions, :].float()
    lg = lg.masked_fill(INVALID_MASK, float("-inf"))
    return torch.topk(torch.log_softmax(lg, dim=-1), cfg.topK, dim=-1).indices.cpu()

def _overlap(a, b):
    """Mean |topK(a) ∩ topK(b)| / K over rows."""
    out = []
    for r in range(a.shape[0]):
        out.append(len(set(a[r].tolist()) & set(b[r].tolist())) / cfg.topK)
    return sum(out) / max(len(out), 1)

@torch.no_grad()
def measure_overlaps(question, solution, states, committed_blocks, s_prompt_len):
    """Returns per-state overlap for teacher A and teacher B."""
    B = cfg.block_size
    s_prompt_ids = tok(build_student_prompt(question), return_tensors="pt").input_ids.to(model.device)
    t_prompt_ids = tok(build_teacher_prompt(question, solution),
                       return_tensors="pt").input_ids.to(model.device)
    t_prompt_len = t_prompt_ids.shape[1]

    # committed prefix per block index (what precedes block b in both constructions)
    rows = []
    for si, st in enumerate(states):
        if si % max(1, cfg.state_stride) != 0:
            continue
        b = st["block"]
        if b >= len(committed_blocks):
            continue
        prior = torch.cat(committed_blocks[:b]).unsqueeze(0) if b > 0 else \
                torch.empty((1, 0), dtype=torch.long, device=model.device)
        cur = torch.full((1, B), MASK_ID, dtype=torch.long, device=model.device)
        for j in st["revealed_before"]:
            cur[0, j] = committed_blocks[b][j]

        # ---- teacher A: privileged prompt, same response state ----
        a_ids = torch.cat([t_prompt_ids, prior, cur], dim=1)
        a_base = t_prompt_len + prior.shape[1]
        a_top = _topk_at(a_ids, t_prompt_len, [a_base + j for j in st["reveal_now"]])

        # ---- teacher B: student prompt + lookahead at future positions ----
        cur_b = cur.clone()
        later = [j for j in range(B)
                 if j not in st["revealed_before"] and j not in st["reveal_now"]]
        for j in later[:cfg.future_reveal_k]:
            cur_b[0, j] = committed_blocks[b][j]
        b_ids = torch.cat([s_prompt_ids, prior, cur_b], dim=1)
        b_base = s_prompt_len + prior.shape[1]
        b_top = _topk_at(b_ids, s_prompt_len, [b_base + j for j in st["reveal_now"]])

        rows.append({"state": si, "block": b, "n_pos": len(st["reveal_now"]),
                     "overlap_A_privileged": _overlap(st["topk"], a_top),
                     "overlap_B_selffuture": _overlap(st["topk"], b_top)})
    return rows

print("overlap measurement ready | K =", cfg.topK, "| future_reveal_k =", cfg.future_reveal_k)

## 10 · Main loop — resumable, one JSONL line per problem

Appends to `overlap_topk.jsonl` after every problem and skips problems already present, so a
Colab disconnect costs at most the problem in flight. Delete the file to start over.

In [ ]:
import json, os, time

done = set()
if RESUME and os.path.exists(LOG_PATH):
    for line in open(LOG_PATH):
        try: done.add(json.loads(line)["qhash"])
        except Exception: pass
    print(f"resuming: {len(done)} problems already measured")

f = open(LOG_PATH, "a")
t_start = time.time()
for i, p in enumerate(opsd_pool[:cfg.n_problems]):
    qh = str(abs(hash(p["question"])) % (10**12))
    if qh in done:
        print(f"[{i+1}/{cfg.n_problems}] skip (done)"); continue
    t0 = time.time()
    states, blocks, plen, text = student_rollout_with_states(p["question"])
    rows = measure_overlaps(p["question"], p["solution"], states, blocks, plen)
    if not rows:
        print(f"[{i+1}/{cfg.n_problems}] no states, skipping"); continue
    mA = sum(r["overlap_A_privileged"] for r in rows) / len(rows)
    mB = sum(r["overlap_B_selffuture"] for r in rows) / len(rows)
    rec = {"qhash": qh, "idx": i, "n_states": len(rows),
           "boxed": "\\boxed" in text,
           "mean_overlap_A_privileged": mA, "mean_overlap_B_selffuture": mB,
           "rows": rows}
    f.write(json.dumps(rec) + "\n"); f.flush()
    done.add(qh)
    print(f"[{i+1}/{cfg.n_problems}] {len(rows)} states | "
          f"A(privileged) {mA:.3f} | B(self-future) {mB:.3f} | gap {mA-mB:+.3f} | "
          f"{time.time()-t0:.0f}s | total {(time.time()-t_start)/60:.1f} min")
f.close()
print("\nDONE ->", LOG_PATH)

## 11 · Aggregate and plot

The headline number is the gap between the two constructions. Read it against the paper's
claim: if A sits near 1 and B sits clearly below, the privileged teacher genuinely had no
signal to give, and the self-future construction does.

In [ ]:
import json, statistics
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in open(LOG_PATH)]
A = [r["mean_overlap_A_privileged"] for r in recs]
Bv = [r["mean_overlap_B_selffuture"] for r in recs]
allA = [x["overlap_A_privileged"] for r in recs for x in r["rows"]]
allB = [x["overlap_B_selffuture"]  for r in recs for x in r["rows"]]

def ci(vals, n=10000, seed=0):
    import random; random.seed(seed); m = len(vals)
    b = sorted(sum(random.choice(vals) for _ in range(m))/m for _ in range(n))
    return b[int(.025*n)], b[int(.975*n)]

print(f"problems: {len(recs)} | states measured: {len(allA)}\n")
for name, v in [("A  privileged prefix (our OPSD-1)", allA),
                ("B  self-future lookahead (OPSD-2)", allB)]:
    lo, hi = ci(v)
    print(f"{name:<38} mean {statistics.mean(v):.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
gap = statistics.mean(allA) - statistics.mean(allB)
print(f"\ngap (A - B): {gap:+.3f}")
print("Interpretation: overlap near 1.0 means the teacher's top-K is the student's top-K,")
print("so the KL target carries no information. Lower is better for distillation.")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(allA, bins=20, range=(0,1), alpha=.75, label="A privileged", color="#cc3311")
ax[0].hist(allB, bins=20, range=(0,1), alpha=.75, label="B self-future", color="#4477aa")
ax[0].set_xlabel("Overlap Top-K (per state)"); ax[0].set_ylabel("# states")
ax[0].legend(frameon=False); ax[0].set_title("distribution over denoising states")
ax[1].plot(range(len(A)), A, "s-", color="#cc3311", label="A privileged", markersize=4)
ax[1].plot(range(len(Bv)), Bv, "o-", color="#4477aa", label="B self-future", markersize=4)
ax[1].set_xlabel("problem"); ax[1].set_ylabel("mean Overlap Top-K")
ax[1].set_ylim(0, 1); ax[1].legend(frameon=False); ax[1].set_title("per-problem means")
plt.tight_layout()
out_png = os.path.join(cfg.drive_root, "fig_overlap_topk.png")
plt.savefig(out_png, dpi=200, bbox_inches="tight"); plt.show()
print("\nwrote", out_png)

json.dump({"n_problems": len(recs), "n_states": len(allA), "topK": cfg.topK,
           "future_reveal_k": cfg.future_reveal_k,
           "mean_overlap_A_privileged": statistics.mean(allA),
           "mean_overlap_B_selffuture": statistics.mean(allB),
           "ci_A": list(ci(allA)), "ci_B": list(ci(allB)), "gap": gap},
          open(os.path.join(cfg.drive_root, "overlap_summary.json"), "w"), indent=1)
print("wrote overlap_summary.json")

## 12 · Bootstrap confidence intervals (Part B)

**Pure CPU. No GPU, no model, no Drive.** Runs anywhere, including locally. Point the path at
`results/floor_grid_per_problem.csv` from the repository (or your own reproduction of it) and
it regenerates every grid cell with a 95% interval.

This is what answers the "n=40 is too small" objection. Note what it shows: the *cliff* is
rock solid (t=4 intervals exclude zero, t=8 intervals are exactly zero, no overlap), while
individual point comparisons such as b16-vs-b32 at t=1 are **not** resolvable at this sample
size. Report both facts.

In [ ]:
import csv, random, os

GRID_CSV = "results/floor_grid_per_problem.csv"     # repo path; edit if running elsewhere
if not os.path.exists(GRID_CSV):
    GRID_CSV = os.path.join(cfg.drive_root, "grid_per_problem_full.csv")

rows = list(csv.DictReader(open(GRID_CSV)))
print(f"loaded {len(rows)} per-problem rows from {GRID_CSV}\n")

def boot_ci(vals, n=10000, seed=0):
    random.seed(seed); m = len(vals)
    b = sorted(sum(random.choice(vals) for _ in range(m))/m for _ in range(n))
    return b[int(.025*n)], b[int(.975*n)]

METRICS = ["correct", "boxed", "repeat4"]
cells = sorted({(r["block_size"], int(r["tokens_per_step"])) for r in rows},
               key=lambda x: (x[0], x[1]))

out = {}
for metric in METRICS:
    print(f"--- {metric} ---")
    print(f"{'cell':<12}{'point':>8}{'95% CI':>20}{'width':>8}{'n':>5}")
    for blk, t in cells:
        vals = [float(r[metric]) for r in rows
                if r["block_size"] == blk and int(r["tokens_per_step"]) == t
                and r[metric] not in ("", "nan")]
        if not vals: continue
        pt = sum(vals)/len(vals); lo, hi = boot_ci(vals)
        out[f"{metric}|b{blk}|t{t}"] = {"point": pt, "ci_low": lo, "ci_high": hi, "n": len(vals)}
        print(f"b{blk} t{t:<8}{pt:>8.3f}   [{lo:.3f}, {hi:.3f}]{hi-lo:>8.3f}{len(vals):>5}")
    print()

json.dump(out, open(os.path.join(cfg.drive_root, "bootstrap_cis.json"), "w"), indent=1)
print("wrote bootstrap_cis.json")

# LaTeX-ready rows for the paper's Table 1
print("\n--- LaTeX (correct rate with CIs, for Table 1) ---")
for blk in sorted({c[0] for c in cells}):
    line = [f"    {blk}"]
    for t in (1, 2, 4, 8, 16):
        k = f"correct|b{blk}|t{t}"
        if k in out:
            d = out[k]
            line.append(f"{d['point']:.3f} \\tiny[{d['ci_low']:.2f},{d['ci_high']:.2f}]")
        else:
            line.append("--")
    print(" & ".join(line) + r" \\")

## 13 · What to do with these numbers

**If A ≈ 1.0 and B is clearly lower**, the paper's Section 4.3 upgrades from an inspection to a
measurement, and gains a positive counterpart: the privileged-prefix construction is the
problem, not self-distillation. Add the overlap figure and cite Luo et al. for the metric and
for their matching finding at 8B.

**If A and B are both near 1.0**, that is also publishable and more interesting for a small
model: at 4B the student is so confident that *no* self-teacher construction can differ from it
enough to teach. Say that plainly. It strengthens the compression-floor framing rather than
weakening it.

**If A is well below 1.0**, the loss collapse observed in OPSD-1 needs a different explanation,
and the honest move is to say the mechanism is unresolved rather than assert the one that the
measurement just contradicted.

For the CIs: put them in Table 1 and Table 2, and replace the Limitations sentence about ±15
points with the measured intervals. Keep the observation that the cliff is resolvable while
individual cells are not, because it tells a reviewer you understand your own power limits.